<a href="https://colab.research.google.com/github/sakshigawade2808/Student-Performance-Prediction/blob/main/Student_Performance_Interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#find .pkl file
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.pkl'):
            print(os.path.join(root, file))

/content/drive/MyDrive/student_performance_random_forest.pkl
/content/drive/MyDrive/student_performance_model_v2.pkl


In [ ]:
import os

model_path = "/content/drive/MyDrive/student_performance_model_v2.pkl"

print(os.path.exists(model_path))

True


In [ ]:
#Load your trained model
import joblib

model = joblib.load(model_path)

print("Model loaded successfully!")
print(type(model))

Model loaded successfully!
<class 'sklearn.pipeline.Pipeline'>


In [ ]:
#Test the model before creating Streamlit
print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Family_Size',
                                                   'Previous_GPA',
                                                   'Number_of_Failed_Courses',
                                                   'Total_Credits_Earned',
                                                   'Weekly_Study_Hours',
                                                   'Attendance_Rate',
                                                   'Library_Visits_Per_Month',
                                                 

In [ ]:
#Install Streamlit
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 53.5 MB/s eta 0:00:00


In [ ]:
!pip install streamlit plotly pyngrok joblib scikit-learn

In [ ]:
!pip install -q plotly joblib

In [ ]:
!pkill -f streamlit

In [ ]:
!streamlit run app.py > streamlit.log 2>&1 &

In [ ]:
!cat streamlit.log

In [ ]:
#Check the app file
!ls -l app.py

ls: cannot access 'app.py': No such file or directory


In [ ]:
!ls -lh app.py

ls: cannot access 'app.py': No such file or directory


In [ ]:
!pkill -f streamlit

In [ ]:
#Install/start ngrok(Since we're running Streamlit inside Google Colab, we need a public tunnel.)
from pyngrok import ngrok

print("ngrok imported successfully!")

ngrok imported successfully!


In [ ]:
#Add your ngrok authentication token(If you already have an ngrok account, get your Auth Token from your ngrok dashboard.)
from pyngrok import ngrok

ngrok.set_auth_token("3KMsZxotEae4hwsk0jnkpnC6vhY_6pSj199mTLPmzngWui6SW")

print("ngrok authentication configured!")

ngrok authentication configured!


In [ ]:
!streamlit run app.py > streamlit.log 2>&1 &

In [ ]:
!cat streamlit.log

In [ ]:
!head -30 app.py

head: cannot open 'app.py' for reading: No such file or directory


In [ ]:
!grep -n "<h1>\|<p>\|Project Goal\|Model Information\|use_container_width" app.py

grep: app.py: No such file or directory


In [ ]:
!rm -f app.py

In [ ]:
!ls -l app.py

ls: cannot access 'app.py': No such file or directory


In [ ]:
!find /content -name "app.py" -type f 2>/dev/null

In [ ]:

%%writefile /content/app.py

import os
import time
import joblib
import numpy as np
import pandas as pd
import streamlit as st

# =========================================================
# 1. APP CONFIGURATION
# =========================================================

MODEL_PATH = "/content/drive/MyDrive/student_performance_model_v2.pkl"

# Change this to your college's official passing requirement.
PASS_MARK = 40.0

st.set_page_config(
    page_title="Student Performance Predictor",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="expanded"
)

# =========================================================
# 2. STYLING — HIGH CONTRAST, READABLE TEXT
# =========================================================

st.markdown("""
<style>
/* Main background and default text */
.stApp {
    background: linear-gradient(135deg, #f5f7ff 0%, #eef4ff 100%);
    color: #1e293b !important;
}

.main .block-container {
    padding-top: 1.5rem;
    padding-bottom: 2rem;
    max-width: 1400px;
}

/* General headings */
h1, h2, h3, h4, h5, h6 {
    color: #172554 !important;
}

/* Normal text */
p, li, label,
[data-testid="stMarkdownContainer"],
[data-testid="stWidgetLabel"],
[data-testid="stCaptionContainer"] {
    color: #334155;
}

/* Dark hero banner */
.hero {
    padding: 28px;
    border-radius: 22px;
    background: linear-gradient(120deg, #172554, #4338ca, #6366f1);
    color: #ffffff !important;
    margin-bottom: 22px;
    box-shadow: 0 8px 24px rgba(49, 46, 129, 0.18);
}

.hero h1 {
    color: #ffffff !important;
    font-size: 2.1rem;
}

.hero p {
    color: #eef2ff !important;
}

/* Metric cards */
.metric-card {
    padding: 20px;
    background: #ffffff;
    border: 1px solid #dbe3f0;
    border-radius: 16px;
    box-shadow: 0 4px 14px rgba(15, 23, 42, 0.05);
    min-height: 125px;
}

.metric-label {
    color: #475569 !important;
    font-size: 0.9rem;
}

.metric-value {
    color: #172554 !important;
    font-size: 1.8rem;
    font-weight: 750;
}

.small-note {
    color: #475569 !important;
    font-size: 0.85rem;
}

/* Roadmap cards */
.roadmap-item {
    padding: 16px;
    background: #ffffff;
    color: #1e293b !important;
    border: 1px solid #dbe3f0;
    border-left: 5px solid #4f46e5;
    border-radius: 12px;
    margin-bottom: 12px;
}

.roadmap-item strong,
.roadmap-item p {
    color: #1e293b !important;
}

/* Pass and fail panels */
.status-pass {
    padding: 18px;
    background: #dcfce7;
    border: 1px solid #86efac;
    border-radius: 14px;
    color: #166534 !important;
}

.status-fail {
    padding: 18px;
    background: #fee2e2;
    border: 1px solid #fca5a5;
    border-radius: 14px;
    color: #991b1b !important;
}

.status-pass h3,
.status-pass p {
    color: #166534 !important;
}

.status-fail h3,
.status-fail p {
    color: #991b1b !important;
}

/* Text and numeric input fields */
.stTextInput input,
.stNumberInput input,
.stTextArea textarea {
    color: #1e293b !important;
    background-color: #ffffff !important;
    border: 1px solid #cbd5e1 !important;
    -webkit-text-fill-color: #1e293b !important;
}

/* Select boxes */
[data-baseweb="select"] > div {
    color: #1e293b !important;
    background-color: #ffffff !important;
}

[data-baseweb="popover"],
[data-baseweb="menu"],
[data-baseweb="list"] {
    background-color: #ffffff !important;
    color: #1e293b !important;
}

[data-baseweb="menu"] *,
[data-baseweb="list"] * {
    color: #1e293b !important;
}

/* Sidebar */
[data-testid="stSidebar"] {
    background: #eef2ff;
}

[data-testid="stSidebar"] * {
    color: #1e293b;
}

/* Buttons */
div.stButton > button {
    color: #ffffff !important;
    background-color: #4338ca !important;
    border: 1px solid #4338ca !important;
    border-radius: 10px;
    font-weight: 650;
}

div.stButton > button:hover {
    color: #ffffff !important;
    background-color: #3730a3 !important;
    border-color: #3730a3 !important;
}

/* Download buttons */
[data-testid="stDownloadButton"] button {
    color:  #FF000000 !important;
    background-color: 	#64748b !important;
}

/* Tables, expanders and tabs */
[data-testid="stDataFrame"],
[data-testid="stExpander"] {
    background: #ffffff;
    border-radius: 10px;
}

button[data-baseweb="tab"] {
    color: #334155 !important;
}

/* Alerts */
[data-testid="stAlert"] {
    color: #1e293b;
}
</style>
""", unsafe_allow_html=True)

# =========================================================
# 3. FEATURE DEFINITIONS
# =========================================================

FALLBACK_FEATURES = [
    "Age",
    "Gender",
    "Region_Type",
    "Family_Size",
    "Major_Subject",
    "Parent_Education",
    "Family_Income_Level",
    "Internet_Quality",
    "Study_Space_Quality",
    "Previous_GPA",
    "Number_of_Failed_Courses",
    "Total_Credits_Earned",
    "Weekly_Study_Hours",
    "Attendance_Rate",
    "Library_Visits_Per_Month",
    "Extracurricular_Hours",
    "Sleep_Hours",
    "Social_Media_Usage_Hours",
    "Stress_Level",
    "Motivation_Score",
    "Self_Efficacy_Score",
    "Midterm_Mark"
]

CATEGORICAL_OPTIONS = {
    "Gender": ["Female", "Male", "Other", "Prefer not to say"],
    "Region_Type": ["Urban", "Rural", "Semi-Urban"],
    "Major_Subject": [
        "Computer Science", "Computer Engineering", "IT",
        "Mechanical", "Civil", "Electrical", "Electronics",
        "Business", "Arts", "Science", "Other"
    ],
    "Parent_Education": [
        "No Formal Education", "Primary", "Secondary",
        "Higher Secondary", "Graduate", "Postgraduate", "Other"
    ],
    "Family_Income_Level": [
        "Low", "Lower-Middle", "Middle", "Upper-Middle", "High"
    ],
    "Internet_Quality": ["Poor", "Average", "Good", "Excellent"],
    "Study_Space_Quality": ["Poor", "Average", "Good", "Excellent"]
}

NUMERIC_SETTINGS = {
    "Age": (15.0, 80.0, 20.0, 1.0),
    "Family_Size": (1.0, 20.0, 4.0, 1.0),
    "Previous_GPA": (0.0, 10.0, 7.0, 0.1),
    "Number_of_Failed_Courses": (0.0, 30.0, 0.0, 1.0),
    "Total_Credits_Earned": (0.0, 300.0, 60.0, 1.0),
    "Weekly_Study_Hours": (0.0, 100.0, 15.0, 1.0),
    "Attendance_Rate": (0.0, 100.0, 80.0, 1.0),
    "Library_Visits_Per_Month": (0.0, 100.0, 4.0, 1.0),
    "Extracurricular_Hours": (0.0, 80.0, 4.0, 1.0),
    "Sleep_Hours": (0.0, 24.0, 7.0, 0.5),
    "Social_Media_Usage_Hours": (0.0, 24.0, 3.0, 0.5),
    "Stress_Level": (1.0, 10.0, 5.0, 1.0),
    "Motivation_Score": (0.0, 10.0, 6.0, 0.5),
    "Self_Efficacy_Score": (0.0, 10.0, 6.0, 0.5),
    "Midterm_Mark": (0.0, 100.0, 60.0, 1.0)
}

FEATURE_LABELS = {
    "Age": "Age",
    "Gender": "Gender",
    "Region_Type": "Region / Area Type",
    "Family_Size": "Family Size",
    "Major_Subject": "Major Subject",
    "Parent_Education": "Parent Education",
    "Family_Income_Level": "Family Income Level",
    "Internet_Quality": "Internet Quality",
    "Study_Space_Quality": "Study Space Quality",
    "Previous_GPA": "Previous GPA",
    "Number_of_Failed_Courses": "Number of Failed Courses",
    "Total_Credits_Earned": "Total Credits Earned",
    "Weekly_Study_Hours": "Weekly Study Hours",
    "Attendance_Rate": "Attendance Rate (%)",
    "Library_Visits_Per_Month": "Library Visits Per Month",
    "Extracurricular_Hours": "Extracurricular Hours Per Week",
    "Sleep_Hours": "Sleep Hours Per Day",
    "Social_Media_Usage_Hours": "Social Media Hours Per Day",
    "Stress_Level": "Stress Level (1–10)",
    "Motivation_Score": "Motivation Score (0–10)",
    "Self_Efficacy_Score": "Self-Efficacy Score (0–10)",
    "Midterm_Mark": "Midterm Mark"
}

CATEGORICAL_FEATURES = set(CATEGORICAL_OPTIONS.keys())

# =========================================================
# 4. LOAD TRAINED MODEL
# =========================================================

@st.cache_resource
def load_model(path, modified_time):
    return joblib.load(path)


model = None
model_error = None

if os.path.exists(MODEL_PATH):
    try:
        model = load_model(MODEL_PATH, os.path.getmtime(MODEL_PATH))
    except Exception as error:
        model_error = str(error)
else:
    model_error = (
        f"Model not found at {MODEL_PATH}. "
        "Mount Google Drive and verify the filename."
    )

# =========================================================
# 5. DETECT MODEL INPUT COLUMNS
# =========================================================

def get_expected_features(loaded_model):
    if loaded_model is None:
        return FALLBACK_FEATURES.copy()

    names = getattr(loaded_model, "feature_names_in_", None)

    if names is not None:
        return list(names)

    named_steps = getattr(loaded_model, "named_steps", {})
    preprocessor = named_steps.get("preprocessor")

    if preprocessor is not None:
        names = getattr(preprocessor, "feature_names_in_", None)
        if names is not None:
            return list(names)

    return FALLBACK_FEATURES.copy()


EXPECTED_FEATURES = get_expected_features(model)


def predict_score(input_df):
    if model is None:
        raise RuntimeError(
            "The model could not be loaded. Check MODEL_PATH."
        )

    missing = [
        column for column in EXPECTED_FEATURES
        if column not in input_df.columns
    ]

    if missing:
        raise ValueError(
            "Missing model input columns: " + ", ".join(missing)
        )

    # Keep exactly the column names and order expected by the model.
    model_input = input_df[EXPECTED_FEATURES].copy()
    prediction = model.predict(model_input)
    score = float(np.asarray(prediction).reshape(-1)[0])

    if not np.isfinite(score):
        raise ValueError("The model returned an invalid score.")

    return score


# =========================================================
# 6. SCORE STATUS AND ROADMAP
# =========================================================

def risk_level(score):
    if score >= 75:
        return "Low", "Predicted score is in the higher range."
    if score >= 60:
        return "Moderate", "Regular revision and practice are recommended."
    if score >= PASS_MARK:
        return "High", "Focused preparation may help improve performance."
    return "Critical", "Predicted score is below the configured pass mark."


def get_value(student, feature):
    return student.get(feature, None)


def build_roadmap(student, score):
    tasks = []

    attendance = get_value(student, "Attendance_Rate")
    study_hours = get_value(student, "Weekly_Study_Hours")
    previous_gpa = get_value(student, "Previous_GPA")
    midterm = get_value(student, "Midterm_Mark")
    failed = get_value(student, "Number_of_Failed_Courses")
    stress = get_value(student, "Stress_Level")
    sleep = get_value(student, "Sleep_Hours")
    motivation = get_value(student, "Motivation_Score")
    social_media = get_value(student, "Social_Media_Usage_Hours")
    library_visits = get_value(student, "Library_Visits_Per_Month")

    if attendance is not None and attendance < 75:
        tasks.append({
            "priority": "High",
            "title": "Improve attendance",
            "action": (
                f"Entered attendance is {attendance:.1f}%. "
                "Set a weekly attendance target, attend scheduled classes, "
                "and catch up on missed notes."
            )
        })

    if study_hours is not None and study_hours < 15:
        tasks.append({
            "priority": "High",
            "title": "Build a consistent study routine",
            "action": (
                f"You entered {study_hours:.1f} study hours per week. "
                "Make a realistic timetable with daily revision, practice "
                "questions, and a weekly review."
            )
        })

    if previous_gpa is not None and previous_gpa < 6:
        tasks.append({
            "priority": "High",
            "title": "Strengthen foundational concepts",
            "action": (
                "Revise prerequisite topics, make short notes, solve basic "
                "questions first, and ask for help with difficult concepts."
            )
        })

    if midterm is not None and midterm < 60:
        tasks.append({
            "priority": "High",
            "title": "Review examination mistakes",
            "action": (
                f"Your entered midterm mark is {midterm:.1f}. "
                "Identify weak chapters, practise previous questions, "
                "and attempt timed practice papers."
            )
        })

    if failed is not None and failed > 0:
        tasks.append({
            "priority": "High",
            "title": "Create a backlog recovery plan",
            "action": (
                f"You entered {int(failed)} failed course(s). List the "
                "pending topics and assign dedicated revision sessions "
                "and weekly practice tests."
            )
        })

    if stress is not None and stress >= 8:
        tasks.append({
            "priority": "Medium",
            "title": "Manage workload and stress",
            "action": (
                "Break tasks into smaller steps, take regular breaks, "
                "and speak with a trusted teacher or counsellor if stress "
                "is affecting daily life."
            )
        })

    if sleep is not None and sleep < 6:
        tasks.append({
            "priority": "Medium",
            "title": "Improve your sleep routine",
            "action": (
                "Maintain a consistent sleep schedule and avoid routinely "
                "reducing sleep to study late at night."
            )
        })

    if motivation is not None and motivation < 5:
        tasks.append({
            "priority": "Medium",
            "title": "Set small weekly goals",
            "action": (
                "Choose two achievable study goals each week and track "
                "your progress. Focus on consistent effort."
            )
        })

    if social_media is not None and social_media > 4:
        tasks.append({
            "priority": "Medium",
            "title": "Reduce distractions",
            "action": (
                "Keep your phone away during focused study sessions and "
                "check social media during planned breaks."
            )
        })

    if library_visits is not None and library_visits < 2:
        tasks.append({
            "priority": "Low",
            "title": "Use additional learning resources",
            "action": (
                "Try library materials, teacher-recommended resources, "
                "or structured online practice for difficult topics."
            )
        })

    if score < PASS_MARK:
        tasks.insert(0, {
            "priority": "Urgent",
            "title": "Prioritise passing requirements",
            "action": (
                "The predicted score is below the configured pass mark. "
                "Confirm assessment requirements with your teacher and "
                "prioritise high-weightage topics and incomplete work."
            )
        })

    if not tasks:
        tasks.append({
            "priority": "Maintain",
            "title": "Maintain your current study habits",
            "action": (
                "Continue regular revision, attend classes, practise "
                "sample papers, and review teacher feedback."
            )
        })

    return tasks


def create_report(student, score, tasks):
    status = "PASS" if score >= PASS_MARK else "FAIL"
    risk, explanation = risk_level(score)

    lines = [
        "STUDENT PERFORMANCE PREDICTION REPORT",
        "=" * 42,
        f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}",
        "",
        f"Predicted score: {score:.2f}",
        f"Configured pass mark: {PASS_MARK:.2f}",
        f"Predicted status: {status}",
        f"Illustrative risk level: {risk}",
        explanation,
        "",
        "STUDENT INPUTS",
        "-" * 30
    ]

    for feature, value in student.items():
        label = FEATURE_LABELS.get(feature, feature)
        lines.append(f"{label}: {value}")

    lines.extend(["", "IMPROVEMENT ROADMAP", "-" * 30])

    for number, task in enumerate(tasks, start=1):
        lines.append(
            f"{number}. {task['title']} [{task['priority']}]"
        )
        lines.append(task["action"])
        lines.append("")

    lines.extend([
        "IMPORTANT NOTE",
        "This is a machine-learning estimate, not a guaranteed exam result. "
        "Pass/fail uses the configured threshold. Roadmap suggestions are "
        "rule-based guidance and do not guarantee improvement."
    ])

    return "\n".join(lines)


# =========================================================
# 7. INPUT CONTROLS
# =========================================================

def render_feature_input(feature, key_prefix):
    label = FEATURE_LABELS.get(feature, feature.replace("_", " "))

    if feature in CATEGORICAL_OPTIONS:
        return st.selectbox(
            label,
            CATEGORICAL_OPTIONS[feature],
            key=f"{key_prefix}_{feature}"
        )

    if feature in NUMERIC_SETTINGS:
        minimum, maximum, default, step = NUMERIC_SETTINGS[feature]
        return st.number_input(
            label,
            min_value=float(minimum),
            max_value=float(maximum),
            value=float(default),
            step=float(step),
            key=f"{key_prefix}_{feature}"
        )

    return st.number_input(
        label,
        value=0.0,
        key=f"{key_prefix}_{feature}"
    )


def collect_student_inputs(key_prefix="student"):
    groups = [
        (
            "👤 Student Information",
            [
                "Age", "Gender", "Region_Type", "Family_Size",
                "Major_Subject", "Parent_Education", "Family_Income_Level"
            ]
        ),
        (
            "📚 Academic Information",
            [
                "Previous_GPA", "Number_of_Failed_Courses",
                "Total_Credits_Earned", "Weekly_Study_Hours",
                "Attendance_Rate", "Library_Visits_Per_Month",
                "Extracurricular_Hours"
            ]
        ),
        (
            "📝 Examination and Learning Information",
            [
                "Midterm_Mark", "Internet_Quality", "Study_Space_Quality",
                "Sleep_Hours", "Social_Media_Usage_Hours", "Stress_Level",
                "Motivation_Score", "Self_Efficacy_Score"
            ]
        )
    ]

    values = {}
    placed = set()

    for heading, features in groups:
        visible = [f for f in features if f in EXPECTED_FEATURES]

        if not visible:
            continue

        st.subheader(heading)
        columns = st.columns(2)

        for index, feature in enumerate(visible):
            with columns[index % 2]:
                values[feature] = render_feature_input(
                    feature, key_prefix
                )
            placed.add(feature)

    # Show any extra features required by the saved model.
    remaining = [f for f in EXPECTED_FEATURES if f not in placed]

    if remaining:
        with st.expander("Other features required by the saved model"):
            columns = st.columns(2)

            for index, feature in enumerate(remaining):
                with columns[index % 2]:
                    values[feature] = render_feature_input(
                        feature, key_prefix
                    )

    return {feature: values[feature] for feature in EXPECTED_FEATURES}


# =========================================================
# 8. DISPLAY RESULTS
# =========================================================

def render_prediction(student_values, score):
    status = "PASS" if score >= PASS_MARK else "FAIL"
    risk, explanation = risk_level(score)
    tasks = build_roadmap(student_values, score)

    st.markdown("---")
    st.header("📊 Prediction Results")

    first, second, third = st.columns(3)

    with first:
        st.markdown(f"""
        <div class="metric-card">
            <div class="metric-label">Predicted Final Exam Score</div>
            <div class="metric-value">{score:.2f}</div>
            <div class="small-note">Out of 100, if your model uses this scale</div>
        </div>
        """, unsafe_allow_html=True)

    with second:
        panel = "status-pass" if status == "PASS" else "status-fail"
        emoji = "✅" if status == "PASS" else "⚠️"

        st.markdown(f"""
        <div class="{panel}">
            <h3>{emoji} Predicted {status}</h3>
            <p>Configured pass mark: {PASS_MARK:.1f}</p>
        </div>
        """, unsafe_allow_html=True)

    with third:
        st.markdown(f"""
        <div class="metric-card">
            <div class="metric-label">Illustrative Risk Level</div>
            <div class="metric-value">{risk}</div>
            <div class="small-note">{explanation}</div>
        </div>
        """, unsafe_allow_html=True)

    if score < PASS_MARK:
        st.error(
            f"The prediction is {PASS_MARK - score:.2f} points below "
            "the configured pass mark. This is an estimate, not a "
            "confirmed failure."
        )
    else:
        st.success(
            f"The prediction is {score - PASS_MARK:.2f} points above "
            "the configured pass mark."
        )

    st.subheader("Score position")
    st.progress(min(max(score, 0.0), 100.0) / 100.0)
    st.caption(
        "The bar uses a 0–100 scale. Confirm that your target variable "
        "uses this scale before interpreting it as a percentage."
    )

    st.markdown("---")
    st.header("🗺️ Personalized Improvement Roadmap")
    st.write(
        "These suggestions are generated from the entered student "
        "information using simple rules."
    )

    for number, task in enumerate(tasks, start=1):
        st.markdown(f"""
        <div class="roadmap-item">
            <strong>{number}. {task['title']}</strong><br>
            <span class="small-note">Priority: {task['priority']}</span>
            <p>{task['action']}</p>
        </div>
        """, unsafe_allow_html=True)

    with st.expander("📅 Suggested 4-week action plan"):
        st.markdown("""
        **Week 1 — Identify weak areas**
        - Review previous marks and teacher feedback.
        - List difficult topics and incomplete assignments.
        - Prepare a realistic weekly timetable.

        **Week 2 — Revise and practise**
        - Revise weak topics in short daily sessions.
        - Solve practice questions after each topic.
        - Ask for help when a concept is unclear.

        **Week 3 — Test your understanding**
        - Attempt a timed practice paper.
        - Review incorrect answers and revise weak topics.
        - Track attendance and study consistency.

        **Week 4 — Review and improve**
        - Attempt another practice test.
        - Compare mistakes with the previous test.
        - Adjust the plan based on teacher feedback.
        """)

    report = create_report(student_values, score, tasks)

    st.download_button(
        "⬇️ Download Student Report",
        data=report,
        file_name="student_performance_report.txt",
        mime="text/plain",
        use_container_width=True
    )

    result_df = pd.DataFrame([student_values])
    result_df["Predicted_Final_Exam_Score"] = score
    result_df["Predicted_Status"] = status
    result_df["Configured_Pass_Mark"] = PASS_MARK
    result_df["Illustrative_Risk_Level"] = risk

    st.download_button(
        "⬇️ Download Prediction as CSV",
        data=result_df.to_csv(index=False).encode("utf-8"),
        file_name="student_prediction.csv",
        mime="text/csv",
        use_container_width=True
    )


# =========================================================
# 9. SIDEBAR
# =========================================================

with st.sidebar:
    st.markdown("## 🎓 Student Predictor")
    st.caption("Academic performance estimation")

    page = st.radio(
        "Navigate",
        [
            "🏠 Home",
            "🔮 Predict Performance",
            "🧪 What-if Analysis",
            "📂 Batch CSV Prediction",
            "ℹ️ About"
        ]
    )

    st.markdown("---")
    st.markdown("### Prediction Settings")
    st.write(f"Configured pass mark: **{PASS_MARK:.1f}**")

    if model is not None:
        st.success("Model loaded successfully")
    else:
        st.error("Model not loaded")


# =========================================================
# 10. HOME PAGE
# =========================================================

if page == "🏠 Home":
    st.markdown("""
    <div class="hero">
        <h1>🎓 Student Performance Predictor</h1>
        <p>
            Estimate a student's final exam score, view the predicted
            pass/fail status, and prepare a practical improvement roadmap.
        </p>
    </div>
    """, unsafe_allow_html=True)

    col1, col2, col3 = st.columns(3)

    with col1:
        st.markdown("""
        <div class="metric-card">
            <div class="metric-label">FEATURE 01</div>
            <div class="metric-value">📈 Score</div>
            <p>Get a numeric final exam score estimate.</p>
        </div>
        """, unsafe_allow_html=True)

    with col2:
        st.markdown("""
        <div class="metric-card">
            <div class="metric-label">FEATURE 02</div>
            <div class="metric-value">🎯 Status</div>
            <p>Compare the estimate with your configured pass mark.</p>
        </div>
        """, unsafe_allow_html=True)

    with col3:
        st.markdown("""
        <div class="metric-card">
            <div class="metric-label">FEATURE 03</div>
            <div class="metric-value">🗺️ Roadmap</div>
            <p>Get study suggestions based on entered information.</p>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("---")
    st.subheader("How to use this application")

    st.markdown("""
    1. Select **Predict Performance** in the sidebar.
    2. Enter student and academic details.
    3. Click **Predict Student Performance**.
    4. Review the predicted score, status, and improvement roadmap.
    5. Download the report if required.
    """)

    st.info(
        "This prediction is an estimate from a trained model. It should "
        "support early academic assistance, not replace teacher judgement "
        "or official examination results."
    )

    if model_error:
        st.error(f"Model loading issue: {model_error}")


# =========================================================
# 11. SINGLE-STUDENT PREDICTION
# =========================================================

elif page == "🔮 Predict Performance":
    st.title("🔮 Predict Student Performance")
    st.write(
        "Enter the student information below. Required inputs are "
        "determined from the saved model's feature names."
    )

    if model is None:
        st.error("The trained model could not be loaded.")
        st.code(model_error or "Unknown model loading error")
        st.stop()

    with st.form("student_prediction_form"):
        student_values = collect_student_inputs("student")
        submitted = st.form_submit_button(
            "🚀 Predict Student Performance",
            type="primary",
            use_container_width=True
        )

    if submitted:
        try:
            student_df = pd.DataFrame(
                [student_values],
                columns=EXPECTED_FEATURES
            )
            score = predict_score(student_df)

            st.session_state["last_student"] = student_values
            st.session_state["last_score"] = score

            st.success("Prediction completed successfully.")
            render_prediction(student_values, score)

        except Exception as error:
            st.error("Prediction could not be completed.")
            st.exception(error)

    elif "last_score" in st.session_state:
        st.info("Showing your most recent prediction.")
        render_prediction(
            st.session_state["last_student"],
            st.session_state["last_score"]
        )


# =========================================================
# 12. WHAT-IF ANALYSIS
# =========================================================

elif page == "🧪 What-if Analysis":
    st.title("🧪 What-if Analysis")
    st.write(
        "Change selected inputs and compare the model's predictions. "
        "This comparison does not prove that changing one factor will "
        "cause the same change in real-world results."
    )

    if model is None:
        st.error("The trained model could not be loaded.")
        st.stop()

    if "last_student" not in st.session_state:
        st.warning(
            "First make a prediction in Predict Performance. "
            "Your latest inputs will then be available here."
        )
    else:
        original = st.session_state["last_student"].copy()
        baseline_score = st.session_state["last_score"]

        st.metric("Original predicted score", f"{baseline_score:.2f}")

        editable_features = [
            feature for feature in [
                "Weekly_Study_Hours",
                "Attendance_Rate",
                "Midterm_Mark",
                "Previous_GPA",
                "Sleep_Hours",
                "Stress_Level"
            ]
            if feature in EXPECTED_FEATURES
        ]

        scenario = original.copy()
        st.subheader("Adjust student inputs")

        for feature in editable_features:
            label = FEATURE_LABELS.get(feature, feature)
            current = float(original[feature])
            low, high, _, step = NUMERIC_SETTINGS[feature]

            scenario[feature] = st.slider(
                label,
                min_value=float(low),
                max_value=float(high),
                value=float(np.clip(current, low, high)),
                step=float(step),
                key=f"whatif_{feature}"
            )

        if st.button(
            "Compare Scenario",
            type="primary",
            use_container_width=True
        ):
            try:
                scenario_df = pd.DataFrame(
                    [scenario],
                    columns=EXPECTED_FEATURES
                )
                scenario_score = predict_score(scenario_df)
                difference = scenario_score - baseline_score

                left, middle, right = st.columns(3)

                left.metric("Original score", f"{baseline_score:.2f}")
                middle.metric(
                    "Scenario score",
                    f"{scenario_score:.2f}",
                    delta=f"{difference:+.2f}"
                )
                right.metric(
                    "Scenario status",
                    "PASS" if scenario_score >= PASS_MARK else "FAIL"
                )

                st.caption(
                    "The difference is between two model predictions. "
                    "It does not guarantee a real student's marks will "
                    "change by this amount."
                )

                if st.button("Save scenario as latest prediction"):
                    st.session_state["last_student"] = scenario
                    st.session_state["last_score"] = scenario_score
                    st.success("Scenario saved for this session.")

            except Exception as error:
                st.error("Scenario prediction failed.")
                st.exception(error)


# =========================================================
# 13. BATCH CSV PREDICTION
# =========================================================

elif page == "📂 Batch CSV Prediction":
    st.title("📂 Batch Student Prediction")
    st.write(
        "Upload a CSV containing the exact input columns expected by "
        "your trained model to predict scores for multiple students."
    )

    if model is None:
        st.error("The trained model could not be loaded.")
        st.stop()

    st.subheader("Required model input columns")
    st.code(", ".join(EXPECTED_FEATURES))

    uploaded_file = st.file_uploader(
        "Upload student CSV",
        type=["csv"]
    )

    if uploaded_file is not None:
        try:
            batch_df = pd.read_csv(uploaded_file)

            st.subheader("Preview uploaded data")
            st.dataframe(batch_df.head(10), use_container_width=True)

            missing = [
                column for column in EXPECTED_FEATURES
                if column not in batch_df.columns
            ]

            if missing:
                st.error(
                    "Missing required columns: " + ", ".join(missing)
                )
                st.info(
                    "Use the exact column names shown above. Extra "
                    "columns are allowed and will be ignored."
                )

            elif st.button(
                "Predict All Students",
                type="primary",
                use_container_width=True
            ):
                input_batch = batch_df[EXPECTED_FEATURES].copy()
                predictions = np.asarray(
                    model.predict(input_batch)
                ).reshape(-1)

                output_df = batch_df.copy()
                output_df["Predicted_Final_Exam_Score"] = predictions
                output_df["Predicted_Status"] = np.where(
                    predictions >= PASS_MARK, "PASS", "FAIL"
                )
                output_df["Illustrative_Risk_Level"] = [
                    risk_level(float(value))[0]
                    for value in predictions
                ]

                st.session_state["batch_results"] = output_df
                st.success(
                    f"Predictions generated for {len(output_df)} students."
                )

            if "batch_results" in st.session_state:
                output_df = st.session_state["batch_results"]

                st.subheader("Batch prediction results")
                st.dataframe(output_df, use_container_width=True)

                c1, c2, c3 = st.columns(3)

                c1.metric("Students processed", len(output_df))
                c2.metric(
                    "Predicted pass",
                    int((output_df["Predicted_Status"] == "PASS").sum())
                )
                c3.metric(
                    "Predicted fail",
                    int((output_df["Predicted_Status"] == "FAIL").sum())
                )

                st.download_button(
                    "⬇️ Download Batch Results",
                    data=output_df.to_csv(index=False).encode("utf-8"),
                    file_name="batch_student_predictions.csv",
                    mime="text/csv",
                    use_container_width=True
                )

        except Exception as error:
            st.error("Could not process the CSV.")
            st.exception(error)


# =========================================================
# 14. ABOUT PAGE
# =========================================================

elif page == "ℹ️ About":
    st.title("ℹ️ About This Application")

    st.markdown("""
    ### Project purpose
    This application estimates a student's final exam score from
    academic, learning, and student-context features.

    ### Machine-learning model
    The app loads your saved scikit-learn model pipeline. It does not
    retrain or overwrite the model when a student submits the form.

    ### Pass/fail calculation
    - **Pass:** predicted score is equal to or above the configured pass mark.
    - **Fail:** predicted score is below the configured pass mark.

    This is a threshold-based estimate, not an official result.

    ### Improvement roadmap
    The roadmap uses rules applied to entered values, including attendance,
    study hours, GPA, midterm marks, failed courses, sleep, stress, and
    social media usage. It is not a separate trained model.

    ### Important limitations
    - Predictions depend on the quality of the training data.
    - The pass mark must match the institution's rules.
    - The target score scale must be verified.
    - Predictions may be inaccurate and should not be used alone to make
      important decisions about students.
    - Protect student information and get appropriate permission before
      uploading or sharing identifiable student records.
    """)

    st.markdown("---")
    st.caption("Student Performance Prediction | Machine Learning Project")

Writing /content/app.py


In [ ]:
!ls -lh /content/app.py

-rw-r--r-- 1 root root 36K Oct 20 05:34 /content/app.py


In [ ]:
!grep -n "use_container_width" /content/app.py

796:        use_container_width=True
810:        use_container_width=True
929:            use_container_width=True
1017:            use_container_width=True
1084:            st.dataframe(batch_df.head(10), use_container_width=True)
1103:                use_container_width=True
1129:                st.dataframe(output_df, use_container_width=True)
1148:                    use_container_width=True


In [ ]:
!grep -n "Project Goal\|Model Information" /content/app.py

In [ ]:
!pkill -9 -f streamlit || true
!pkill -9 -f ngrok || true

^C
^C


In [ ]:
!streamlit run /content/app.py --server.port 8502 > /content/streamlit_new.log 2>&1 &

In [ ]:
!cat /content/streamlit_new.log

In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8502)

print("NEW STREAMLIT URL:")
print(public_url)

NEW STREAMLIT URL:
NgrokTunnel: "https://cadet-monsieur-devotee.ngrok-free.dev" -> "http://localhost:8502"
